# 🥈 Silver Layer — Transform & Cleanse

> **Purpose**: Apply business rules, type casting, deduplication, and SCD Type 2 to produce
> a clean, trusted dataset ready for Gold aggregations.

## Steps
1. Read from Bronze Delta
2. Cast types & handle nulls
3. Window-based deduplication
4. Quarantine invalid records
5. SCD Type 2 merge into Silver

In [ ]:
# ── Parameters ────────────────────────────────────────────────────────────
dbutils.widgets.text('source_table',   'dbo.sales',        'Source Bronze Table')
dbutils.widgets.text('storage_account','<storage_account>','ADLS Gen2 Account Name')
dbutils.widgets.text('batch_id',       '',                 'Batch ID')

source_table    = dbutils.widgets.get('source_table')
storage_account = dbutils.widgets.get('storage_account')
batch_id        = dbutils.widgets.get('batch_id') or spark.sparkContext.applicationId

import re
table_safe  = re.sub(r'[^a-zA-Z0-9_]', '_', source_table).strip('_')
bronze_path = f'abfss://bronze@{storage_account}.dfs.core.windows.net/{table_safe}'
silver_path = f'abfss://silver@{storage_account}.dfs.core.windows.net/{table_safe}'
quarantine_path = f'abfss://quarantine@{storage_account}.dfs.core.windows.net/{table_safe}'

print(f'Bronze : {bronze_path}')
print(f'Silver : {silver_path}')

In [ ]:
# ── Read Bronze ────────────────────────────────────────────────────────────
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

df_bronze = spark.read.format('delta').load(bronze_path)
print(f'Bronze rows: {df_bronze.count():,}')

In [ ]:
# ── Type Casting & Null Handling ───────────────────────────────────────────
# Adjust column names and types to match your actual schema
df_typed = (
    df_bronze
    # Cast core columns
    .withColumn('sale_id',     F.col('sale_id').cast('string'))
    .withColumn('customer_id', F.col('customer_id').cast('string'))
    .withColumn('product_id',  F.col('product_id').cast('string'))
    .withColumn('sale_date',   F.to_date(F.col('sale_date'), 'yyyy-MM-dd'))
    .withColumn('quantity',    F.col('quantity').cast('integer'))
    .withColumn('unit_price',  F.col('unit_price').cast('decimal(18,4)'))
    .withColumn('region',      F.upper(F.trim(F.col('region'))))
    # Fill nulls with safe defaults
    .fillna({'quantity': 0, 'unit_price': 0.0, 'region': 'UNKNOWN'})
    # Derive revenue
    .withColumn('revenue',     F.round(F.col('quantity') * F.col('unit_price'), 2))
    # Add Silver audit
    .withColumn('_silver_processed_at', F.current_timestamp())
    .withColumn('_silver_batch_id',     F.lit(batch_id))
)

print(f'Typed rows: {df_typed.count():,}')

In [ ]:
# ── Data Quality — Split Good / Quarantine ────────────────────────────────
df_valid = df_typed.filter(
    (F.col('sale_id').isNotNull()) &
    (F.col('quantity') > 0) &
    (F.col('unit_price') > 0) &
    (F.col('sale_date').isNotNull()) &
    (F.col('sale_date') <= F.current_date())
)

df_quarantine = df_typed.subtract(df_valid)

print(f'Valid rows      : {df_valid.count():,}')
print(f'Quarantined rows: {df_quarantine.count():,}')

# Persist quarantine for review
if df_quarantine.count() > 0:
    (
        df_quarantine
        .withColumn('_quarantine_reason', F.lit('failed_dq_checks'))
        .write.format('delta').mode('append').save(quarantine_path)
    )
    print('Quarantine records written')

In [ ]:
# ── Window-Based Deduplication ─────────────────────────────────────────────
# Keep the latest version of each sale_id based on _ingested_at
window_spec = Window.partitionBy('sale_id').orderBy(F.col('_ingested_at').desc())

df_deduped = (
    df_valid
    .withColumn('_row_num', F.row_number().over(window_spec))
    .filter(F.col('_row_num') == 1)
    .drop('_row_num')
)

print(f'After dedup: {df_deduped.count():,} rows')

In [ ]:
# ── SCD Type 2 Merge into Silver ──────────────────────────────────────────
# Add SCD2 tracking columns
df_silver_new = (
    df_deduped
    .withColumn('_effective_from', F.current_timestamp())
    .withColumn('_effective_to',   F.lit(None).cast('timestamp'))
    .withColumn('_is_current',     F.lit(True))
)

if DeltaTable.isDeltaTable(spark, silver_path):
    dt_silver = DeltaTable.forPath(spark, silver_path)
    (
        dt_silver.alias('target')
        .merge(
            df_silver_new.alias('source'),
            'target.sale_id = source.sale_id AND target._is_current = true'
        )
        # Expire changed rows
        .whenMatchedUpdate(
            condition='target.revenue <> source.revenue OR target.quantity <> source.quantity',
            set={ '_is_current': 'false', '_effective_to': 'source._effective_from' }
        )
        .whenNotMatchedInsertAll()
        .execute()
    )
    # Insert new versions of changed rows
    changed = df_silver_new.join(
        spark.table(f'silver.{table_safe}').filter(~F.col('_is_current')),
        'sale_id', 'left_semi'
    )
    if changed.count() > 0:
        changed.write.format('delta').mode('append').save(silver_path)
    print('SCD2 merge complete')
else:
    (
        df_silver_new.write
        .format('delta').mode('overwrite')
        .option('path', silver_path)
        .saveAsTable(f'silver.{table_safe}')
    )
    print(f'Silver table created: {df_silver_new.count():,} rows')

spark.sql(f'OPTIMIZE silver.{table_safe}')
print('Silver layer complete')